# Пара 12. Справочные данные (Wikidata, Open Library) и проверка ответа ИИ

| | |
|---|---|
| **Какой вопрос диссертации решает** | «Как быстро собрать фактическую канву (даты, издания, переводы)? Как пользоваться чатом ИИ, не попадая на выдуманные цитаты?» |
| **Что вы получите** | факты и список произведений автора, издания и переводы, промпт для бесплатного чата и автоматическую проверку его цитат |
| **Трек** | ваш автор (демо — Бунин) · 80–90 мин |

**Правило курса:** вопрос → процедура → картинка → **цитата** → ограничение. Число без цитаты в диссертацию не идёт.

### Как работать (3 правила)
1. **Файл → Сохранить копию на Диске.** Иначе ваши правки пропадут.
2. Запускайте ячейки **по очереди сверху вниз** кнопкой ▶ слева от ячейки. Код спрятан — менять нужно **только поля формы** (справа в ячейке).
3. Сначала запустите всё **как есть** — это демо на Бунине, результат заведомо осмысленный. Потом в ячейке «Тексты» выберите **«Мои файлы»** и пройдите те же шаги на материале своей диссертации.

> Что-то сломалось? **Среда выполнения → Перезапустить сеанс**, затем ячейки сверху вниз.
> Красная ошибка — скопируйте **последние 3 строки** преподавателю или в бесплатный чат ИИ с вопросом «что это значит для новичка?».
> Свои файлы — **.txt (UTF-8) или .docx**. PDF и .doc сначала пересохраните.

In [ ]:
#@title 🔧 Подготовка — просто запустите ▶ (≈20 сек) { display-mode: "form" }
import requests, importlib
_u = "https://pavelaleks.github.io/samarkand_lectures/demo-downloads/colab-bunin/dh_utils.py"
open("dh_utils.py", "w", encoding="utf-8").write(requests.get(_u, timeout=60).text)
import dh_utils; importlib.reload(dh_utils)
from dh_utils import *
setup()

## Часть 1. Wikidata — факты и произведения

In [ ]:
#@title 🔎 Найти автора в Wikidata { display-mode: "form" }
AUTHOR = "Иван Бунин"  #@param {type:"string"}
found = wikidata_find(AUTHOR)
found

In [ ]:
#@title 📋 Факты и произведения { display-mode: "form" }
#@markdown Впишите QID из таблицы выше (строка с правильным описанием). Пусто — возьмём первую строку.
QID = ""  #@param {type:"string"}
_f = globals().get("found")
qid = QID.strip() or (_f.iloc[0]["QID"] if _f is not None and not _f.empty else "")
if not qid:
    print("Сначала найдите автора в ячейке выше или впишите QID вручную.")
else:
    facts, works = wikidata_author(qid)
    display(facts)
    display(works.head(30))

## Часть 2. Open Library — издания и переводы

In [ ]:
#@title 📚 Издания { display-mode: "form" }
AUTHOR_LATIN = "Ivan Bunin"  #@param {type:"string"}
ol = openlibrary_author(AUTHOR_LATIN)
if not ol.empty and ol["год первого издания"].notna().any():
    import matplotlib.pyplot as plt
    s = ol["год первого издания"].dropna().astype(int)
    ax = s.value_counts().sort_index().plot(kind="bar", figsize=(10, 3), color="#0f766e")
    ax.set_title(f"Open Library: год первого издания · {AUTHOR_LATIN}"); plt.tight_layout(); plt.show()
ol.head(20)

## Часть 3. Бесплатный чат ИИ — с проверкой
Платные модели не нужны: подойдёт любой бесплатный чат (DeepSeek, Qwen, Gemini, GigaChat).
1. Ячейка ниже соберёт **промпт**, который требует дословных цитат.
2. Скопируйте его в чат, ответ вставьте во вторую ячейку.
3. Блокнот проверит, **есть ли эти цитаты в тексте на самом деле**.

In [ ]:
#@title 📂 Тексты { display-mode: "form" }
SOURCE = "Бунин (демо)"  #@param ["Бунин (демо)", "Чехов (демо)", "Мои файлы"]
#@markdown **Для демо:** названия через «;» или слово `все`. Полный список — запустите `list_demo("Бунин (демо)")` в новой ячейке.
STORIES = "Чистый понедельник"  #@param {type:"string"}
#@markdown **Для своих файлов:** оставьте поле пустым — появится кнопка «Выбрать файлы».
MY_FILES = ""  #@param {type:"string"}
texts = load_texts(SOURCE, STORIES, MY_FILES)

In [ ]:
#@title 🤖 Собрать промпт { display-mode: "form" }
QUESTION = "Как в этом фрагменте сталкиваются сакральное и светское пространство?"  #@param {type:"string"}
START_WORD = 0  #@param {type:"integer"}
N_WORDS = 600  #@param {type:"slider", min:200, max:900, step:50}
_t = list(texts.values())[0].split()
fragment = " ".join(_t[START_WORD:START_WORD + N_WORDS])
_ = llm_prompt(fragment, QUESTION)

In [ ]:
#@title ✅ Проверить цитаты в ответе ИИ { display-mode: "form" }
#@markdown Вставьте ответ модели целиком между тройными кавычками (в самом коде ячейки: дважды щёлкните по заголовку, чтобы открыть код).
ANSWER = """
Пример ответа: 1) Рассказчик каждый вечер везёт героиню мимо храма: «Каждый вечер мчал меня в этот час на вытягивающемся рысаке мой кучер» (уверенность высокая).
2) Героиня признаётся: «я ещё не знаю, во что верю» (уверенность средняя).
3) Она «ходит в кремлёвские соборы по вечерам» (уверенность высокая).
"""
check_quotes(ANSWER, texts)

### Протокол работы с ИИ (вставьте в приложение диссертации)
1. Модель и дата: _ · 2. Фрагмент: _ · 3. Промпт: _ · 4. Ответ: _
5. **Сверка:** сколько цитат найдено дословно / частично / не найдено.
6. Что перешло в диссертацию — **только** после сверки по печатному изданию; формулировки — ваши.

### 🎓 Как это выглядит в диссертации
**Абзац для главы (образец — перепишите под себя):**
> Фактические сведения (даты, издания, переводы) сверены по Wikidata (_) и Open Library (дата обращения _). Чат-модель _ использовалась для генерации гипотез по фрагментам текста; все приведённые ею цитаты проверены автоматически на наличие в тексте и вручную по изданию _ (из _ цитат подтверждено _).

**На защите — один слайд:** схема «вопрос → ответ ИИ → проверка цитат → что осталось» — комиссии важно увидеть, что ИИ под контролем.

**Когда метод врёт (впишите в «Ограничения»):** Wikidata и Open Library неполны и содержат ошибки; ИИ уверенно выдумывает цитаты, даты и источники; «дословно найдено» ≠ «правильно истолковано».

### ✅ Что сдать с пары
- [ ] таблица фактов/произведений вашего автора (или абзац, почему Wikidata вам не помогла)
- [ ] промпт + ответ ИИ + таблица проверки цитат
- [ ] заполненный протокол работы с ИИ

Всё складывайте в свой **«Методологический паспорт диссертации»** (шаблон — на сайте курса, раздел «Применить к диссертации»).